<a href="https://colab.research.google.com/github/tinemyumi/saude-mental-datasus/blob/main/notebooks/4.consolidacao_base_analitica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Base Analítica de Saúde Mental**

**Objetivo:** Construir base no nível município × ano para análise de acesso, desigualdade e oferta de serviços em saúde mental.

**Autor:** Larissa Tinem

# **1. Importação e configuração**

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

# Paleta para visualizações rápidas
sns.set_theme(style='whitegrid', palette='muted')

from google.colab import drive
drive.mount('/content/drive')

print('Ambiente pronto.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ambiente pronto.


# **2. Carregamento dos dados**

In [2]:
df = pd.read_parquet('/content/drive/MyDrive/Dados/dados_tratados/df_sihsus.parquet')
df_pop = pd.read_csv('/content/drive/MyDrive/Dados/dados_tratados/df_populacao_sp')
df_drs = pd.read_csv('/content/drive/MyDrive/Dados/dados_tratados/df_drs.csv')
df_estabelecimentos = pd.read_csv('/content/drive/MyDrive/Dados/dados_tratados/df_estabelecimentos.csv')

In [6]:
# ============================================================
# BASE ANALÍTICA PARA TESTES ESTATÍSTICOS
# Objetivo:
# - Agregar internações por município, ano e mês
# - Juntar população e DRS
# - Calcular taxa de internação por 100 mil habitantes
# - Criar período pré / pandemia / pós considerando ano e mês
# ============================================================

import pandas as pd
import numpy as np

# 1. PADRONIZAR CÓDIGOS DOS MUNICÍPIOS

# Padroniza município de residência
df['MUNIC_RES'] = df['MUNIC_RES'].astype(str).str.zfill(6)

# Padroniza código do município na população
df_pop['cod_municipio'] = df_pop['cod_municipio'].astype(str).str.zfill(6)

# Padroniza código do município na base DRS
df_drs['cod_municipio'] = df_drs['cod_municipio'].astype(str).str.zfill(6)

# Garante que ano e mês sejam numéricos
df['ANO_CMPT'] = pd.to_numeric(df['ANO_CMPT'], errors='coerce').astype('Int64')
df['MES_CMPT'] = pd.to_numeric(df['MES_CMPT'], errors='coerce').astype('Int64')
df_pop['ano'] = pd.to_numeric(df_pop['ano'], errors='coerce').astype('Int64')

# Garante que MORTE seja numérica
df['MORTE'] = pd.to_numeric(df['MORTE'], errors='coerce').fillna(0).astype(int)

# Garante que DIAS_PERM seja numérica
df['DIAS_PERM'] = pd.to_numeric(df['DIAS_PERM'], errors='coerce')

# Filtra apenas municípios de SP
df = df[df['MUNIC_RES'].str.startswith('35')].copy()

# 2. AGREGAR INTERNAÇÕES POR MUNICÍPIO, ANO E MÊS

base_analitica = (
    df.groupby(['MUNIC_RES', 'ANO_CMPT', 'MES_CMPT'])
      .agg(
          n_internacoes=('N_AIH', 'count'),       # Conta internações
          n_mortes=('MORTE', 'sum'),             # Soma mortes
          media_dias_perm=('DIAS_PERM', 'mean')  # Média de dias de permanência
      )
      .reset_index()
)

# Renomeia colunas
base_analitica = base_analitica.rename(columns={
    'MUNIC_RES': 'cod_municipio',
    'ANO_CMPT': 'ano',
    'MES_CMPT': 'mes'
})

# 3. CRIAR DATA MENSAL E PERÍODO DA PANDEMIA

# Cria uma coluna de data usando ano e mês
base_analitica['data_mes'] = pd.to_datetime(
    base_analitica['ano'].astype(str) + '-' +
    base_analitica['mes'].astype(str).str.zfill(2) + '-01',
    errors='coerce'
)

# Classifica o período corretamente
base_analitica['periodo_pandemia'] = np.select(
    [
        base_analitica['data_mes'] < pd.Timestamp('2020-03-01'),
        base_analitica['data_mes'].between(
            pd.Timestamp('2020-03-01'),
            pd.Timestamp('2022-05-01')
        ),
        base_analitica['data_mes'] >= pd.Timestamp('2022-06-01')
    ],
    [
        'pre',
        'pandemia',
        'pos'
    ],
    default=None
)

# 4. JUNTAR POPULAÇÃO

base_analitica = base_analitica.merge(
    df_pop[['cod_municipio', 'ano', 'populacao']],
    on=['cod_municipio', 'ano'],
    how='left'
)

# 5. JUNTAR INFORMAÇÕES DA DRS

base_analitica = base_analitica.merge(
    df_drs[['cod_municipio', 'municipio', 'cod_drs', 'drs']],
    on='cod_municipio',
    how='left'
)

# 6. CALCULAR TAXAS

# Taxa de internação por 100 mil habitantes
base_analitica['taxa_internacao'] = (
    base_analitica['n_internacoes'] / base_analitica['populacao']
) * 100000

# Taxa de mortalidade por 100 mil habitantes
base_analitica['taxa_mortalidade'] = (
    base_analitica['n_mortes'] / base_analitica['populacao']
) * 100000

# 7. LIMPEZA DA BASE

# Remove registros sem população
base_analitica = base_analitica.dropna(subset=['populacao'])

# Remove população zerada ou inválida
base_analitica = base_analitica[base_analitica['populacao'] > 0]

# Remove registros sem taxa
base_analitica = base_analitica.dropna(subset=['taxa_internacao'])

# Remove registros sem período
base_analitica = base_analitica.dropna(subset=['periodo_pandemia'])

# 8. ORGANIZAR COLUNAS

base_analitica = base_analitica[
    [
        'cod_municipio',
        'municipio',
        'ano',
        'mes',
        'data_mes',
        'cod_drs',
        'drs',
        'periodo_pandemia',
        'populacao',
        'n_internacoes',
        'taxa_internacao',
        'n_mortes',
        'taxa_mortalidade',
        'media_dias_perm'
    ]
]

# 9. VERIFICAR RESULTADO

print('Base analítica criada com sucesso!')
print('Linhas:', base_analitica.shape[0])
print('Colunas:', base_analitica.shape[1])

display(base_analitica.head())

# Ver distribuição dos períodos
display(base_analitica['periodo_pandemia'].value_counts())

Base analítica criada com sucesso!
Linhas: 64337
Colunas: 14


,cod_municipio,municipio,ano,mes,data_mes,cod_drs,drs,periodo_pandemia,populacao,n_internacoes,taxa_internacao,n_mortes,taxa_mortalidade,media_dias_perm
0,350010,Adamantina,2015,1,2015-01-01,5.0000,DRS Marília,pre,"35,048.0000",95,271.0568,0,0.0000,30.5158
1,350010,Adamantina,2015,2,2015-02-01,5.0000,DRS Marília,pre,"35,048.0000",87,248.2310,0,0.0000,27.7816
2,350010,Adamantina,2015,3,2015-03-01,5.0000,DRS Marília,pre,"35,048.0000",100,285.3230,0,0.0000,30.6500
3,350010,Adamantina,2015,4,2015-04-01,5.0000,DRS Marília,pre,"35,048.0000",95,271.0568,0,0.0000,30.0421
4,350010,Adamantina,2015,5,2015-05-01,5.0000,DRS Marília,pre,"35,048.0000",91,259.6439,0,0.0000,31.3956


,count
periodo_pandemia,
pre,30799
pos,21136
pandemia,12402


# **3. Merge com caps**

In [7]:
# 1. IDENTIFICAR CAPS NA BASE DE ESTABELECIMENTOS (CNES)

# Códigos oficiais de CAPS no CNES
CAPS_TP_UNIDADE = [70, 71, 72, 73, 74, 75, 76]

# Garantir que TP_UNIDADE está limpo e numérico
df_estabelecimentos['TP_UNIDADE'] = (
    df_estabelecimentos['TP_UNIDADE']
    .astype(str)                        # converte para string
    .str.extract(r'(\d+)')[0]           # extrai apenas números
    .astype(float)                      # converte para float
    .astype('Int64')                    # inteiro com suporte a nulos
)

# Filtrar apenas estabelecimentos que são CAPS
df_caps = df_estabelecimentos[
    df_estabelecimentos['TP_UNIDADE'].isin(CAPS_TP_UNIDADE)
].copy()

print('Quantidade de registros CAPS:', len(df_caps))


# 2. GARANTIR PADRONIZAÇÃO DO MUNICÍPIO

df_caps['cod_municipio'] = (
    df_caps['cod_municipio']
    .astype(str)
    .str.zfill(6)
)


# 3. CONTAR CAPS POR MUNICÍPIO

caps_municipio = (
    df_caps.groupby('cod_municipio')
    .size()
    .reset_index(name='n_caps')
)

print('Municípios com CAPS:', caps_municipio.shape[0])


# 4. INTEGRAR CAPS NA BASE ANALÍTICA

base_analitica['cod_municipio'] = (
    base_analitica['cod_municipio']
    .astype(str)
    .str.zfill(6)
)

base_analitica = base_analitica.merge(
    caps_municipio,
    on='cod_municipio',
    how='left'
)

# Municípios sem CAPS recebem 0
base_analitica['n_caps'] = base_analitica['n_caps'].fillna(0)


# 5. CRIAR VARIÁVEIS DERIVADAS

# CAPS por 100 mil habitantes
base_analitica['caps_por_100k'] = (
    base_analitica['n_caps'] / base_analitica['populacao']
) * 100000


Quantidade de registros CAPS: 1760
Municípios com CAPS: 464


In [8]:
base_analitica.to_parquet('/content/drive/MyDrive/Dados/base_analitica/base_analitica.parquet')